# 04 - Retrieval Evaluation Ablation

Evaluates dense-only, BM25-only, and hybrid retrieval on the locked `gold_benchmark_v1.csv`. This notebook measures document-level and article-level retrieval quality before any generation or fine-tuning.

In [ ]:
from pathlib import Path
import json
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('Not running in Google Colab; using local filesystem paths.')

DRIVE_ROOT = Path('/content/drive/MyDrive/rag')
sys.path.insert(0, str(DRIVE_ROOT))

config = json.loads((DRIVE_ROOT / 'project_config.json').read_text(encoding='utf-8'))
benchmark_csv = DRIVE_ROOT / config['benchmark_csv']
index_root = DRIVE_ROOT / config.get('official_index_root', 'indexes/official_law_v3')

for path in [benchmark_csv, index_root / 'index_manifest.json']:
    if not path.exists():
        raise FileNotFoundError(path)

benchmark_csv, index_root

In [ ]:
import importlib.util
import subprocess
import sys

required_modules = {
    'sentence_transformers': 'sentence-transformers',
    'faiss': 'faiss-cpu',
    'rank_bm25': 'rank-bm25',
    'tqdm': 'tqdm',
}

missing_packages = [package for module, package in required_modules.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('All retrieval dependencies are already installed.')

In [ ]:
import torch
from src.evaluation_retrieval import evaluate_retrieval

device = 'cuda' if torch.cuda.is_available() else 'cpu'
top_k = 10
candidate_k = config['retrieval_defaults']['top_k_retrieval']
dense_weight = config['retrieval_defaults']['dense_weight']
bm25_weight = config['retrieval_defaults']['bm25_weight']

device, top_k, candidate_k, dense_weight, bm25_weight

In [ ]:
summaries = []
for mode in ['dense', 'bm25', 'hybrid']:
    summary = evaluate_retrieval(
        benchmark_csv=benchmark_csv,
        index_root=index_root,
        output_predictions_csv=DRIVE_ROOT / f'outputs/retrieval_eval/{mode}_retrieval_predictions_v1.csv',
        output_summary_json=DRIVE_ROOT / f'outputs/retrieval_eval/{mode}_retrieval_summary_v1.json',
        mode=mode,
        top_k=top_k,
        candidate_k=candidate_k,
        dense_weight=dense_weight,
        bm25_weight=bm25_weight,
        device=device,
    )
    summaries.append(summary)

summaries

In [ ]:
import pandas as pd

rows = []
for summary in summaries:
    row = {'mode': summary['mode'], 'question_count': summary['question_count']}
    row.update(summary['metrics'])
    rows.append(row)

summary_df = pd.DataFrame(rows)
summary_path = DRIVE_ROOT / 'outputs/retrieval_eval/retrieval_ablation_summary_v1.csv'
summary_df.to_csv(summary_path, index=False, encoding='utf-8-sig')
summary_df

In [ ]:
hybrid_predictions = pd.read_csv(DRIVE_ROOT / 'outputs/retrieval_eval/hybrid_retrieval_predictions_v1.csv', dtype=str, keep_default_na=False)
misses = hybrid_predictions[hybrid_predictions['article_hit@5'].astype(float).eq(0)]
print('Hybrid article_hit@5 misses:', len(misses))
misses[['question_id', 'topic', 'difficulty', 'gold_article_keys', 'retrieved_citations_top10']].head(15)

Expected outputs:

- `outputs/retrieval_eval/dense_retrieval_predictions_v1.csv`
- `outputs/retrieval_eval/bm25_retrieval_predictions_v1.csv`
- `outputs/retrieval_eval/hybrid_retrieval_predictions_v1.csv`
- `outputs/retrieval_eval/*_retrieval_summary_v1.json`
- `outputs/retrieval_eval/retrieval_ablation_summary_v1.csv`

Use these metrics to decide whether reranking should be added before generation.